# Random-Vocab Null Control on Llama-3.1-8B

Probe-artifact control: generate "documents" by drawing tokens directly from the Llama vocabulary with **zero structure**, run them through the same corpus_expansion pipeline (100ctx + 30tgt × 3 positions), and verify the heavy-tailed kernel does NOT appear.

Two variants:
- `random_vocab_uniform`: uniform random sampling from the full vocabulary. Maximum entropy, no resemblance to natural text.
- `random_vocab_freq`: sample from natural-corpus token frequency distribution (uses gutenberg_fiction_en token frequencies as the source). Has natural unigram statistics but zero order structure.

**Predictions:**
- alpha_ord ≈ 0 (no marginal benefit from more random context).
- Fit may fail entirely (no positive marginals to fit a power law to).
- If natural-language cells show alpha_ord ≈ −1.3 with the same probe + same pipeline, but random-vocab gives ≈ 0, the heavy tail can't be a probe artifact.

**Output:** `My Drive/LRTIA/Results/corpus_expansion/llama/random_vocab_<variant>.json` (same dir as the natural-language cells so the reconcile script picks them up alongside).

In [ ]:
!pip install -q -U bitsandbytes>=0.46.1 accelerate

import numpy as np
import json, math, random, time
from pathlib import Path
from collections import Counter
from scipy import stats
from tqdm.auto import tqdm
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from google.colab import drive
drive.mount('/content/drive')

DRIVE = Path('/content/drive/MyDrive/LRTIA')
BASE = DRIVE / 'Results/corpus_expansion/llama'
BASE.mkdir(parents=True, exist_ok=True)

MODEL_NAME = 'unsloth/Meta-Llama-3.1-8B'
C = 100              # context length
TARGET_LEN = 30
TARGET_FRACS = [0.25, 0.50, 0.75]
DOCS_PER_VARIANT = 60   # match per-cell N of the en quartet
DOC_TOK_LEN = 2000       # match natural corpus token-count scale
N_SHUFFLES = 1
SEED = 20260503

# Source corpus for frequency distribution. Any natural en cache will do.
FREQ_SOURCE_TXT = DRIVE / 'Data/corpus_expansion/gutenberg_fiction_en'

RUN_VARIANTS = ['random_vocab_uniform', 'random_vocab_freq']

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Variants: {RUN_VARIANTS}')
print(f'Per variant: {DOCS_PER_VARIANT} docs × {DOC_TOK_LEN} tokens × 3 targets')

In [ ]:
# Load Llama (same model + quant config as corpus_expansion).
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb, device_map='auto'
)
model.eval()
VOCAB_SIZE = tokenizer.vocab_size
print(f'Model loaded. vocab_size = {VOCAB_SIZE}')

In [ ]:
# Pipeline functions — identical to Corpus_Expansion_Llama so cache files merge cleanly.
@torch.no_grad()
def ppl_nll(ctx_toks, tgt_toks):
    if len(tgt_toks) < 2:
        return float('inf'), float('inf')
    full = list(ctx_toks) + list(tgt_toks)
    ts = len(ctx_toks)
    ids = torch.tensor([full], device=model.device)
    out = model(ids); logits = out.logits[0]
    nll = 0.0; cnt = 0
    for i in range(ts, len(full) - 1):
        lp = torch.log_softmax(logits[i], dim=-1)
        nll += -lp[full[i + 1]].item(); cnt += 1
    del out, logits; torch.cuda.empty_cache()
    if cnt == 0:
        return float('inf'), float('inf')
    mn = nll / cnt
    return math.exp(mn), mn

def compute_corrected_curves(ctx, tgt):
    mc = len(ctx)
    o_ppl, o_nll, s_ppl, s_nll = [], [], [], []
    for c in range(mc + 1):
        pfx = ctx[-c:] if c > 0 else []
        p, n = ppl_nll(pfx, tgt)
        o_ppl.append(p); o_nll.append(n)
        if c == 0:
            s_ppl.append(p); s_nll.append(n)
        else:
            rng = random.Random(SEED + c)
            sp_l, sn_l = [], []
            for _ in range(N_SHUFFLES):
                sh = list(pfx); rng.shuffle(sh)
                sp, sn = ppl_nll(sh, tgt)
                if not math.isinf(sp):
                    sp_l.append(sp); sn_l.append(sn)
            s_ppl.append(np.mean(sp_l) if sp_l else p)
            s_nll.append(np.mean(sn_l) if sn_l else n)
    dists = list(range(1, mc + 1))
    mo = [o_ppl[d - 1] - o_ppl[d] for d in dists]
    ms = [s_ppl[d - 1] - s_ppl[d] for d in dists]
    delta = [a - b for a, b in zip(mo, ms)]
    mo_n = [o_nll[d - 1] - o_nll[d] for d in dists]
    ms_n = [s_nll[d - 1] - s_nll[d] for d in dists]
    delta_n = [a - b for a, b in zip(mo_n, ms_n)]
    return {
        'distances': dists,
        'ordered_ppl': o_ppl, 'shuffled_ppl': s_ppl, 'delta_ppl': delta,
        'ordered_nll': o_nll, 'shuffled_nll': s_nll, 'delta_nll': delta_n,
    }

BIN_EDGES = [1, 2, 3, 4, 5, 7, 10, 15, 20, 30, 50, 75, 100]
def fit_pl(marg):
    bm, bc = [], []
    for i in range(len(BIN_EDGES) - 1):
        lo, hi = BIN_EDGES[i], BIN_EDGES[i + 1]
        vals = marg[lo - 1:hi - 1]
        vals = vals[~np.isnan(vals)]
        if len(vals) > 0 and np.mean(vals) > 0:
            bm.append(np.mean(vals)); bc.append((lo + hi) / 2)
    if len(bm) >= 4:
        s, _, r, _, _ = stats.linregress(np.log(bc), np.log(bm))
        return s, r
    return None, None

print('Pipeline functions ready')

In [ ]:
# Build the natural-token frequency distribution from a real corpus.
# We use gutenberg_fiction_en raw text files, tokenize them with Llama, count.
# This gives the freq-matched random variant a realistic unigram distribution.
freq_counter = Counter()
src_files = list(FREQ_SOURCE_TXT.glob('*.txt'))[:30]   # 30 files is plenty for unigram stats
for fp in tqdm(src_files, desc='counting tokens'):
    text = fp.read_text(encoding='utf-8', errors='replace')
    ids = tokenizer.encode(text, add_special_tokens=False)
    freq_counter.update(ids)

freq_ids = np.array(list(freq_counter.keys()), dtype=np.int64)
freq_weights = np.array(list(freq_counter.values()), dtype=np.float64)
freq_weights = freq_weights / freq_weights.sum()
print(f'Built freq distribution from {len(src_files)} files: {len(freq_ids)} unique tokens')

In [ ]:
# Generators for the two variants.
def gen_uniform_doc(rng):
    return rng.integers(0, VOCAB_SIZE, size=DOC_TOK_LEN, dtype=np.int64).tolist()

def gen_freq_doc(rng):
    idx = rng.choice(len(freq_ids), size=DOC_TOK_LEN, p=freq_weights)
    return freq_ids[idx].tolist()

GEN = {
    'random_vocab_uniform': gen_uniform_doc,
    'random_vocab_freq': gen_freq_doc,
}

# Sanity: sample one short doc from each, decode for visibility.
rng = np.random.default_rng(SEED)
for variant, gen in GEN.items():
    sample = gen(rng)[:30]
    print(f'{variant} sample (first 30 tokens):')
    print(f'  tokens: {sample}')
    print(f'  decoded: {tokenizer.decode(sample)[:200]!r}')
    print()

In [ ]:
# Main run loop — one cache JSON per variant, same format as corpus_expansion.
for variant in RUN_VARIANTS:
    cache_path = BASE / f'{variant}.json'
    if cache_path.exists():
        with open(cache_path) as f: n = len(json.load(f))
        print(f'\n{variant}: cached ({n})'); continue

    print(f'\n{"="*60}\n{variant}: {DOCS_PER_VARIANT} docs × 3 targets = {DOCS_PER_VARIANT * 3} target slots\n{"="*60}')

    rng = np.random.default_rng(SEED)
    gen = GEN[variant]
    t0 = time.time()
    results = []

    for doc_idx in tqdm(range(DOCS_PER_VARIANT), desc=variant):
        full_ids = gen(rng)
        n_tok = len(full_ids)
        rem_start = C
        rem_end = n_tok - TARGET_LEN
        if rem_end <= rem_start:
            continue

        for frac in TARGET_FRACS:
            ts = int(rem_start + frac * (rem_end - rem_start))
            te = ts + TARGET_LEN
            cs = ts - C
            ce = ts
            ctx = full_ids[cs:ce]
            tgt = full_ids[ts:te]
            r = compute_corrected_curves(ctx, tgt)
            r['corpus_id'] = variant
            r['document_id'] = f'{variant}__doc{doc_idx:03d}'
            r['target_id'] = f'{variant}__doc{doc_idx:03d}__pos{int(frac*100):02d}'
            r['target_frac'] = frac
            r['language'] = 'synthetic'
            r['family'] = 'synthetic'
            r['genre'] = 'random_vocab'
            r['modality'] = 'synthetic'
            results.append(r)

    elapsed = time.time() - t0
    with open(cache_path, 'w') as f:
        json.dump(results, f)
    print(f'  {len(results)} results in {elapsed/60:.1f} min')

    if results:
        mean_delta = float(np.mean([np.mean(r['delta_ppl']) for r in results]))
        # Both ordered and corrected alphas — for natural cells these align;
        # for synthetic noise both should be ~0 or fail to fit.
        ord_curve = np.mean([
            [r['ordered_ppl'][i] - r['ordered_ppl'][i+1] for i in range(len(r['ordered_ppl'])-1)]
            for r in results], axis=0)
        delta_curve = np.mean([r['delta_ppl'] for r in results], axis=0)
        a_o, r_o = fit_pl(np.array(ord_curve))
        a_c, r_c = fit_pl(np.array(delta_curve))
        ostr = f'{a_o:.3f} (r={r_o:.3f})' if a_o else 'fit failed'
        cstr = f'{a_c:.3f} (r={r_c:.3f})' if a_c else 'fit failed'
        print(f'  Mean Δ: {mean_delta:.4f}')
        print(f'  alpha_ordered: {ostr}')
        print(f'  alpha_corrected: {cstr}')

print('\nDone.')